# DE-LU Day-Ahead Forecasting: Full Code Notebook

I wrote this notebook in very easy language so I can understand the full flow quickly.
This is the full main project code, with simple notes about what and why.

## Reading Notes

I wrote these notes in very easy language and in my own voice, so I and others can quickly understand what is happening and why.

For a side-by-side explanation of what each stage does, where it lives, and why it exists, open:

- docs/pipeline_notes.md

This notebook contains executable code blocks, and the notes file is the companion guide.

## 1) Configuration (src/config.py)

In this part, I load my API key safely and stop early if it is missing.

In [ ]:
"""I keep project configuration loading in one place."""

from __future__ import annotations

import os

from dotenv import load_dotenv


def get_entsoe_api_key() -> str:
    """I read ENTSO-E API credentials from the local environment.

    I load variables from .env and fail fast with a clear message when
    ENTSOE_API_KEY is empty or missing.
    """
    # I first load the .env file so local secret values become available.
    load_dotenv()

    # I read and clean the key value.
    api_key = os.getenv("ENTSOE_API_KEY", "").strip()

    # I stop early with a clear message if the key is missing.
    if not api_key:
        raise RuntimeError(
            "Missing ENTSOE_API_KEY. Add it to your .env file (see .env.example)."
        )

    return api_key

## 2) I/O Utilities (src/utils_io.py)

Here I keep small helper functions to read and save files in one consistent way.

In [ ]:
"""I use these helpers for filesystem and Parquet reads/writes."""

from pathlib import Path

import pandas as pd


def ensure_dir(path: str | Path) -> Path:
    """I create a directory if needed and return it as a Path."""
    # I make sure the folder exists before any file write.
    dir_path = Path(path)
    dir_path.mkdir(parents=True, exist_ok=True)
    return dir_path


def save_parquet(df: pd.DataFrame, path: str | Path) -> None:
    """I save a DataFrame to Parquet and create parent folders first."""
    # I always create the parent folder first to avoid write errors.
    file_path = Path(path)
    ensure_dir(file_path.parent)
    df.to_parquet(file_path, index=False)


def load_parquet(path: str | Path) -> pd.DataFrame:
    """I load and return a DataFrame from a Parquet file."""
    file_path = Path(path)
    return pd.read_parquet(file_path)

## 3) Smoke Test (src/smoke_test.py)

I use this quick check to confirm my setup works before running bigger steps.

In [ ]:
"""I run a basic smoke test for project configuration."""


def smoke_main() -> None:
    """I verify that configuration loading works end to end."""
    # I load the key to confirm my local setup is ready.
    _ = get_entsoe_api_key()
    print("Config OK")

## 4) Fetch + Clean Prices (src/fetch_prices.py)

In this stage, I download data and clean it so later steps are reliable.

In [ ]:
from entsoe import EntsoePandasClient
import logging

LOGGER = logging.getLogger(__name__)
TARGET_TZ = "Europe/Berlin"
AREA_CODE = "DE_LU"
RAW_PATH = Path("data/raw/prices_de_lu_raw.parquet")
CLEAN_PATH = Path("data/processed/prices_de_lu_hourly.parquet")


def configure_logging() -> None:
    """I configure a consistent log format for pipeline runs."""
    logging.basicConfig(
        level=logging.INFO,
        format="%(asctime)s | %(levelname)s | %(name)s | %(message)s",
    )


def compute_time_window(years: int = 5) -> tuple[pd.Timestamp, pd.Timestamp]:
    """I compute a [start, end) window for the last `years` in Berlin time."""
    # I keep timestamps aligned to exact hours.
    end = pd.Timestamp.now(tz=TARGET_TZ).floor("h")
    start = end - pd.DateOffset(years=years)
    return start, end


def fetch_raw_prices(start: pd.Timestamp, end: pd.Timestamp) -> pd.Series:
    """I fetch raw day-ahead prices for DE-LU from ENTSO-E."""
    # I create the API client with my secret key.
    api_key = get_entsoe_api_key()
    client = EntsoePandasClient(api_key=api_key)

    # I request prices for the DE-LU bidding zone.
    series = client.query_day_ahead_prices(country_code=AREA_CODE, start=start, end=end)
    if series.empty:
        raise RuntimeError("ENTSO-E returned an empty price series for the requested window.")
    return series


def normalize_timezone(index: pd.DatetimeIndex) -> pd.DatetimeIndex:
    """I normalize timestamps to Europe/Berlin while preserving true instants."""
    if index.tz is None:
        LOGGER.warning("Received naive timestamps; assuming UTC before converting to Europe/Berlin.")
        return index.tz_localize("UTC").tz_convert(TARGET_TZ)
    return index.tz_convert(TARGET_TZ)


def clean_prices(series: pd.Series) -> pd.DataFrame:
    """I clean hourly prices and resolve duplicates deterministically."""
    # I convert bad values to NaN so I can clean them safely.
    df = pd.DataFrame({"price_eur_mwh": pd.to_numeric(series, errors="coerce")})
    df.index = normalize_timezone(df.index)
    df.index.name = "datetime"
    df = df.sort_index()

    # I keep only one value when the same timestamp appears more than once.
    duplicate_mask = df.index.duplicated(keep="last")
    if int(duplicate_mask.sum()) > 0:
        df = df[~duplicate_mask]

    # I drop rows that still have missing prices.
    df = df.dropna(subset=["price_eur_mwh"])
    return df.reset_index()


def fetch_main() -> None:
    """I run fetch, cleaning, and save steps for DE-LU day-ahead prices."""
    configure_logging()
    start, end = compute_time_window(years=5)
    raw_series = fetch_raw_prices(start=start, end=end)

    # I save the raw version for traceability.
    raw_df = pd.DataFrame(
        {
            "datetime": normalize_timezone(raw_series.index),
            "price_eur_mwh": pd.to_numeric(raw_series.values, errors="coerce"),
        }
    )
    raw_df = raw_df.sort_values("datetime").reset_index(drop=True)

    ensure_dir(RAW_PATH.parent)
    ensure_dir(CLEAN_PATH.parent)
    save_parquet(raw_df, RAW_PATH)

    # I save the cleaned version for modeling.
    clean_df = clean_prices(raw_series)
    save_parquet(clean_df, CLEAN_PATH)

## 5) Validate Time Series (src/validate_timeseries.py)

I check data quality here, like missing hours or duplicate timestamps.

In [ ]:
INPUT_PATH = Path("data/processed/prices_de_lu_hourly.parquet")


def validate(df: pd.DataFrame) -> None:
    """I run core time-series checks and log what I find."""
    # I standardize datetime values first.
    ts = df.copy()
    ts["datetime"] = pd.to_datetime(ts["datetime"], errors="coerce", utc=False)
    ts = ts.sort_values("datetime").reset_index(drop=True)

    dt = ts["datetime"]
    idx = pd.DatetimeIndex(dt)

    # I build an hourly reference index to detect missing timestamps.
    full_index = pd.date_range(start=idx.min(), end=idx.max(), freq="h", tz=idx.tz)
    missing_hours = full_index.difference(idx)

    LOGGER.info("Rows: %d", len(ts))
    LOGGER.info("Datetime monotonic increasing: %s", bool(dt.is_monotonic_increasing))
    LOGGER.info("Duplicate timestamps: %d", int(dt.duplicated(keep=False).sum()))
    LOGGER.info("Missing hourly timestamps: %d", len(missing_hours))


def validate_main() -> None:
    """I load cleaned prices and execute validation checks."""
    configure_logging()
    validate(load_parquet(INPUT_PATH))

## 6) Build Features (src/build_features.py)

I transform raw price history into model-ready features and a target column.

In [ ]:
MODEL_DATASET_PATH = Path("data/processed/model_dataset.parquet")


def build_feature_table(prices: pd.DataFrame) -> pd.DataFrame:
    """I create leakage-safe features and a delivery-price target."""
    # I standardize datetime and order rows before feature creation.
    df = prices.copy()
    df["datetime"] = pd.to_datetime(df["datetime"], errors="coerce", utc=False)
    df = df.sort_values("datetime").drop_duplicates(subset=["datetime"], keep="last")
    df = df.set_index("datetime")
    y = pd.to_numeric(df["price_eur_mwh"], errors="coerce")

    # I add simple calendar features.
    feat = pd.DataFrame(index=df.index)
    feat["hour"] = feat.index.hour
    feat["weekday"] = feat.index.weekday
    feat["month"] = feat.index.month
    feat["dayofyear"] = feat.index.dayofyear
    feat["is_weekend"] = (feat["weekday"] >= 5).astype(int)

    # I add lag features to capture recent history.
    feat["lag_24"] = y.shift(24)
    feat["lag_48"] = y.shift(48)
    feat["lag_72"] = y.shift(72)
    feat["lag_168"] = y.shift(168)

    # I compute rolling features from past-known values only.
    past = y.shift(24)
    feat["roll_mean_24"] = past.rolling(window=24, min_periods=24).mean()
    feat["roll_std_24"] = past.rolling(window=24, min_periods=24).std()
    feat["roll_mean_168"] = past.rolling(window=168, min_periods=168).mean()
    feat["roll_std_168"] = past.rolling(window=168, min_periods=168).std()

    # This is the value I want to predict.
    feat["target_price_eur_mwh"] = y
    return feat.dropna().reset_index().rename(columns={"index": "datetime"})


def build_main() -> None:
    # I load cleaned hourly prices, build features, then save dataset.
    df = load_parquet(Path("data/processed/prices_de_lu_hourly.parquet"))
    dataset = build_feature_table(df)
    save_parquet(dataset, MODEL_DATASET_PATH)

## 7) Split Data (src/split_data.py)

I split by time, not random rows, so validation is closer to real forecasting.

In [ ]:
TRAIN_PATH = Path("data/processed/train.parquet")
VALID_PATH = Path("data/processed/valid.parquet")


def split_train_valid(df: pd.DataFrame, valid_days: int = 180) -> tuple[pd.DataFrame, pd.DataFrame]:
    """I split data into historical train and most-recent validation windows."""
    # I use a time-based split to mimic real forecasting.
    out = df.copy()
    out["datetime"] = pd.to_datetime(out["datetime"], errors="coerce", utc=False)
    out = out.sort_values("datetime").reset_index(drop=True)
    cutoff = out["datetime"].max() - pd.Timedelta(days=valid_days)
    return out[out["datetime"] < cutoff].copy(), out[out["datetime"] >= cutoff].copy()


def split_main() -> None:
    dataset = load_parquet(MODEL_DATASET_PATH)
    train, valid = split_train_valid(dataset, valid_days=180)
    save_parquet(train, TRAIN_PATH)
    save_parquet(valid, VALID_PATH)

## 8) Train Baseline (src/train_baseline.py)

I train a simple baseline model and save metrics to measure starting performance.

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
import numpy as np
import json

PRED_PATH = Path("reports/baseline_valid_predictions.parquet")
METRICS_PATH = Path("reports/baseline_metrics.json")
TARGET_COL = "target_price_eur_mwh"


def _build_pipeline(feature_columns: list[str]) -> Pipeline:
    # I one-hot encode calendar categories and impute missing values safely.
    categorical = [c for c in ["hour", "weekday", "month", "is_weekend"] if c in feature_columns]
    numeric = [c for c in feature_columns if c not in categorical]
    preprocessor = ColumnTransformer(
        transformers=[
            (
                "cat",
                Pipeline(steps=[("imputer", SimpleImputer(strategy="most_frequent")), ("onehot", OneHotEncoder(handle_unknown="ignore"))]),
                categorical,
            ),
            (
                "num",
                Pipeline(steps=[("imputer", SimpleImputer(strategy="median"))]),
                numeric,
            ),
        ]
    )
    return Pipeline(steps=[("preprocessor", preprocessor), ("model", LinearRegression())])


def train_main() -> None:
    # I load train and validation datasets.
    train = load_parquet(TRAIN_PATH).sort_values("datetime").reset_index(drop=True)
    valid = load_parquet(VALID_PATH).sort_values("datetime").reset_index(drop=True)

    feature_columns = [c for c in train.columns if c not in {"datetime", TARGET_COL}]
    X_train, y_train = train[feature_columns], train[TARGET_COL]
    X_valid, y_valid = valid[feature_columns], valid[TARGET_COL]

    # I train a simple baseline model.
    model = _build_pipeline(feature_columns)
    model.fit(X_train, y_train)
    y_pred = model.predict(X_valid)

    # I save predictions so I can inspect errors later.
    pred_df = valid[["datetime", "hour", TARGET_COL]].copy().rename(columns={TARGET_COL: "y_true"})
    pred_df["y_pred"] = y_pred
    pred_df["abs_error"] = (pred_df["y_true"] - pred_df["y_pred"]).abs()
    save_parquet(pred_df, PRED_PATH)

    # I save simple quality metrics (MAE and RMSE).
    metrics = {
        "overall": {
            "mae": float(mean_absolute_error(y_valid, y_pred)),
            "rmse": float(np.sqrt(mean_squared_error(y_valid, y_pred))),
        },
        "train_rows": int(len(train)),
        "valid_rows": int(len(valid)),
    }
    METRICS_PATH.parent.mkdir(parents=True, exist_ok=True)
    METRICS_PATH.write_text(json.dumps(metrics, indent=2), encoding="utf-8")

## 9) Run End-to-End (src/run_pipeline.py)

I run all steps in order from setup to training in one command.

In [ ]:
def run_all_main() -> None:
    # I run each step in order because every step depends on the previous one.
    configure_logging()
    smoke_main()
    fetch_main()
    validate_main()
    build_main()
    split_main()
    train_main()
    print("Pipeline finished successfully.")

## 10) Execute (optional)

I run the next cell only after setting ENTSOE_API_KEY in my .env file.

In [ ]:
# I uncomment this when my .env file is ready.
# run_all_main()